# Universal transformer: training

One shared transformer layer looped a fixed number of times (no ACT, no coordinate embeddings, no positional embeddings), trained on multi-hop modular addition.

This notebook:
1. loads and inspects the dataset,
2. builds the UT from its entry in `configs/train.yaml` and looks at it before training,
3. trains it for the full run (or resumes / reloads an existing run in `runs/`).

Analysis lives in `run-analysis.ipynb`: point it at the run folder this notebook writes.

Training logs to TensorBoard as it goes. To watch it live, run `uv run tensorboard --logdir runs` and open http://localhost:6006 (the **Custom Scalars** tab overlays the loss family and per-hop accuracies).

Rerunning the notebook never retrains a finished run: it loads it instead.

In [1]:
# setup: run from the repo root so config paths resolve, and import the code in src/
import json
import os
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import torch as t
import yaml
from IPython.display import Image, display

import models
import train

print("repo root:", ROOT)
print("torch", t.__version__, "| cuda:", t.cuda.is_available())

repo root: /home/saafetensors/Workspace/time-loops
torch 2.14.1+cu130 | cuda: True


## Parameters

`RUN_NAME` picks an entry from `configs/train.yaml`. Use `ut_statenorm_k1_T8_p113_h3_x6` for the variant with an RMS norm on the carried state.

Set `MAX_STEPS` to a small number for a quick demo; demos are written under `runs/demos/` so they never block the full run.

In [2]:
RUN_NAME = "ut_k1_T8_p113_h3_x6"
MAX_STEPS = None
CONFIG = "configs/train.yaml"
OUTPUT_ROOT = None  # None uses the config's output_root (runs/)
DEVICE = "cuda" if t.cuda.is_available() else "cpu"

In [3]:
cfg = yaml.safe_load(Path(CONFIG).read_text())
if OUTPUT_ROOT:
    cfg["output_root"] = OUTPUT_ROOT
train.validate_config(cfg)
entry = next(r for r in cfg["runs"] if r["name"] == RUN_NAME)
print(yaml.safe_dump(entry, sort_keys=False))

name: ut_k1_T8_p113_h3_x6
data: p113_h3_x6
seed: 0
model:
  pos: none
  norm: none
  n_prelude: 0
  k: 1
  core_shared: true
  injection: none
  state_init: embed
  n_coda: 0
depth:
  rule: fixed
  T: 8
train:
  steps: 40000
  eval_every: 100
  sweep_every: 1000
  eval_T_max: 32



## Data

The dataset is generated by `src/make_data.py`. Each row is a chain `x_0 x_1 … x_h =` whose answer is the sum mod p; `intermediates` holds the left-to-right partial sums.

In [4]:
data_dir = Path(cfg["data_root"]) / entry["data"]
data = train.load_data(data_dir, DEVICE)
meta = data["meta"]
c = meta["config"]
print(f"dataset {entry['data']}: p = {c['p']}, trained hops 1..{c['max_train_hops']}, "
      f"tested up to {c['max_test_hops']} hops, generator {meta['generator_version']}")
print(f"hop-1 pair split: {meta['hop1']['n_train']} train, {meta['hop1']['n_val']} val, "
      f"{meta['hop1']['n_heldout']} held out (of {meta['hop1']['space']})")

dataset p113_h3_x6: p = 113, trained hops 1..3, tested up to 6 hops, generator 3.0.0
hop-1 pair split: 3830 train, 1276 val, 7663 held out (of 12769)


In [5]:
# rows per file and group; systematic groups are hop.k<novel step>
for name, groups in data["sets"].items():
    sizes = ", ".join(f"{label}: {len(y)}" for label, (x, y) in groups.items())
    print(f"{name:<18} {sizes}")

train              1: 3830, 2: 15000, 3: 15000
val                1: 1276, 2: 2000, 3: 2000
extrap_depth       4: 2000, 5: 2000, 6: 2000
systematic         2.k1: 1000, 2.k2: 1000, 3.k1: 667, 3.k2: 667, 3.k3: 666
systematic_extrap  4.k1: 500, 4.k2: 500, 4.k3: 500, 4.k4: 500, 5.k1: 400, 5.k2: 400, 5.k3: 400, 5.k4: 400, 5.k5: 400, 6.k1: 334, 6.k2: 334, 6.k3: 333, 6.k4: 333, 6.k5: 333, 6.k6: 333


In [6]:
# a few raw rows from each file
for name in ["train", "val", "extrap_depth", "systematic"]:
    path = data_dir / f"{name}.jsonl"
    if path.exists():
        with open(path) as f:
            lines = f.readlines()
        print(f"--- {name} ({len(lines)} rows)")
        for line in lines[:: max(1, len(lines) // 3)][:3]:
            print(" ", line.strip())

--- train (33830 rows)
  {"hop": 1, "operands": [59, 41], "tokens": ["59", "41", "="], "answer": 100, "intermediates": [100]}
  {"hop": 2, "operands": [102, 80, 44], "tokens": ["102", "80", "44", "="], "answer": 0, "intermediates": [69, 0]}
  {"hop": 3, "operands": [7, 99, 32, 24], "tokens": ["7", "99", "32", "24", "="], "answer": 49, "intermediates": [106, 25, 49]}
--- val (5276 rows)
  {"hop": 1, "operands": [65, 58], "tokens": ["65", "58", "="], "answer": 10, "intermediates": [10]}
  {"hop": 2, "operands": [75, 9, 25], "tokens": ["75", "9", "25", "="], "answer": 109, "intermediates": [84, 109]}
  {"hop": 3, "operands": [92, 84, 65, 53], "tokens": ["92", "84", "65", "53", "="], "answer": 68, "intermediates": [63, 15, 68]}
--- extrap_depth (6000 rows)
  {"hop": 4, "operands": [97, 21, 45, 92, 41], "tokens": ["97", "21", "45", "92", "41", "="], "answer": 70, "intermediates": [5, 50, 29, 70]}
  {"hop": 5, "operands": [111, 7, 33, 45, 107, 48], "tokens": ["111", "7", "33", "45", "107", "

In [7]:
# vocab: number n has id n, and "=" comes last
vocab = json.loads((data_dir / "vocab.json").read_text())
print("vocab size:", len(vocab), "| '=' id:", vocab["="])
print("token id == value for every number:", all(vocab[str(i)] == i for i in range(c["p"])))

# answers should be close to uniform over [0, p) within each hop
for label, (x, y) in data["sets"]["train"].items():
    counts = Counter(y.tolist())
    print(f"train hop {label}: prompt length {x.shape[1]}, {len(counts)} distinct answers, "
          f"per-answer count {min(counts.values())}..{max(counts.values())}")

vocab size: 114 | '=' id: 113
token id == value for every number: True
train hop 1: prompt length 3, 113 distinct answers, per-answer count 23..45
train hop 2: prompt length 4, 113 distinct answers, per-answer count 88..184
train hop 3: prompt length 5, 113 distinct answers, per-answer count 77..182


## Model

`train.resolve` turns the config entry into concrete settings using the dataset's metadata. Below: the settings, parameter counts, and what the untrained model's state looks like over the loop. The size of the state (its root-mean-square) after each pass shows whether it grows across iterations.

In [8]:
mcfg, rule, grad_iters, tr, op = train.resolve(entry, meta)
T = rule.eval_depth()
print(mcfg)
print(f"depth rule: {type(rule).__name__}, training depth T = {T}, grad_iters = {grad_iters}")
print("optimizer:", op)
print("training:", tr)

t.manual_seed(entry["seed"])
model = models.LoopedTransformer(mcfg).to(DEVICE)
by_part = Counter()
for name, prm in model.named_parameters():
    by_part[name.split(".")[0]] += prm.numel()
print("parameters:", dict(by_part), "| total", sum(by_part.values()))

ModelConfig(p=113, pos='none', n_ctx=0, norm='none', n_prelude=0, k=1, core_shared=True, n_core_copies=1, injection='none', state_init='embed', n_coda=0, state_norm=False, d_model=128, num_heads=4, d_head=32, d_mlp=512)
depth rule: FixedDepth, training depth T = 8, grad_iters = None
optimizer: {'lr': 0.001, 'weight_decay': 1.0, 'betas': [0.9, 0.98], 'warmup_steps': 10}
training: {'n_checkpoints': 25, 'chunk_size': 8192, 'resume_every': 1000, 'steps': 40000, 'eval_every': 100, 'sweep_every': 1000, 'eval_T_max': 32}
parameters: {'embed': 14592, 'core': 197248, 'unembed': 14464} | total 226304


In [9]:
# hook every activation on a few val prompts and look at the iteration tags
cache = {}
model.cache_all(cache)
x, y = data["sets"]["val"]["3"]
with t.no_grad():
    logits = model(x[:64], T)
model.remove_all_hooks()

tags = {}
for name, it in cache:
    tags.setdefault(name, set()).add(it)
for name in ["hook_embed", "core.0.0.attn.hook_attn", "core.0.0.mlp.hook_post", "hook_state"]:
    print(f"{name:<26} iterations {sorted(tags[name], key=lambda i: -1 if i is None else i)}")

rms = [cache[("hook_state", i)].pow(2).mean().sqrt().item() for i in range(T)]
print("state rms by pass:", [round(v, 2) for v in rms])
print("logit std at init:", round(logits.std().item(), 2))

hook_embed                 iterations [None]
core.0.0.attn.hook_attn    iterations [0, 1, 2, 3, 4, 5, 6, 7]
core.0.0.mlp.hook_post     iterations [0, 1, 2, 3, 4, 5, 6, 7]
hook_state                 iterations [0, 1, 2, 3, 4, 5, 6, 7]
state rms by pass: [0.19, 0.53, 1.72, 6.04, 22.17, 81.72, 299.35, 1086.47]
logit std at init: 1007.22


## Train

Trains the full run into `runs/<RUN_NAME>/` (or `runs/demos/<RUN_NAME>/` with `MAX_STEPS`). If the run already finished it is loaded as is; if it was interrupted it resumes from `latest.pt`. Progress is printed every 1000 steps. Every evaluation goes to `metrics.jsonl` and to TensorBoard event files in `tb/`.

In [10]:
root = Path(cfg["output_root"]) / ("demos" if MAX_STEPS else "")
run_dir = root / RUN_NAME

def show_every_1000(line):
    """print only the 1000-step progress lines and resume notices."""
    if not line.startswith("step") or int(line.split()[1]) % 1000 == 0:
        print(line, flush=True)

if (run_dir / "checkpoints" / "final.pt").exists():
    print(f"{run_dir} is complete; loading it")
else:
    resume = (run_dir / "latest.pt").exists()
    run_dir = train.train(entry, cfg, DEVICE, max_steps=MAX_STEPS, resume=resume, log=show_every_1000)
print("run directory:", run_dir)
print(json.dumps({k: v for k, v in json.loads((run_dir / "meta.json").read_text()).items()
                  if k in ("git", "device", "steps", "max_steps")}, indent=1))

runs/ut_k1_T8_p113_h3_x6 is complete; loading it
run directory: runs/ut_k1_T8_p113_h3_x6
{
 "git": {
  "commit": "380ac71caf682cda89c3174ccd7769dc43b304ea",
  "dirty": true
 },
 "device": "NVIDIA GeForce RTX 4090",
 "steps": 40000,
 "max_steps": null
}


In [11]:
# quick check of the final weights at the training depth; the full analysis is in run-analysis.ipynb
final = run_dir / "checkpoints" / "final.pt"
model.load_state_dict(t.load(final, map_location=DEVICE))
model.eval()
for name in ("train", "val"):
    res = train.evaluate(model, data["sets"][name], T, tr["chunk_size"])
    print(f"{name:<6}", "  ".join(f"hop {label}: {v['acc']:.3f}" for label, v in res.items()))
print(f"\nanalyse with run-analysis.ipynb, RUN_DIR = {str(run_dir)!r}")

train  hop 1: 0.990  hop 2: 0.664  hop 3: 0.705
val    hop 1: 0.000  hop 2: 0.048  hop 3: 0.031

analyse with run-analysis.ipynb, RUN_DIR = 'runs/ut_k1_T8_p113_h3_x6'
